# 模块 1 · 行为与留存分析

数据来源：`ecommerce_analysis` 库里阶段 1 SQL 产出的 `analysis_*` 结果表（REES46 多品类电商行为数据，2019-10 ~ 2019-11 抽样）。

本 notebook 只做**读取 + 可视化 + 结论**，不重算指标——重计算都在 `sql/` 里完成。

顺序：行为分布 → 转化漏斗 → 加购流失 → 用户留存 → 复购留存 → RFM 分层。

> 内核请选 **Python (sqleco)**。

In [ ]:
import os, getpass
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

# ============ 统一专业制图风格 ============
INK, MUTED, GRID = '#22333B', '#6B7A82', '#E8EAED'
PRIMARY, ACCENT = '#2A6F97', '#E07A5F'
FUNNEL_COLORS = ['#2A6F97', '#468FAF', '#89C2D9']

mpl.rcParams.update({
    'figure.facecolor': 'white', 'axes.facecolor': 'white', 'savefig.facecolor': 'white',
    'figure.dpi': 110, 'savefig.dpi': 150,
    'font.sans-serif': ['Microsoft YaHei', 'SimHei', 'DejaVu Sans'],  # 中文不显示成方块
    'axes.unicode_minus': False,
    'axes.edgecolor': MUTED, 'axes.labelcolor': INK, 'text.color': INK,
    'xtick.color': MUTED, 'ytick.color': MUTED,
    'axes.grid': True, 'axes.axisbelow': True, 'grid.color': GRID, 'grid.linewidth': 0.9,
    'axes.spines.top': False, 'axes.spines.right': False,
    'font.size': 11, 'legend.frameon': False,
})

def titled(ax, title, subtitle=None):
    """左对齐主标题 + 灰色副标题（编辑风），替代默认居中 title。"""
    ax.set_title('')
    ax.text(0, 1.07, title, transform=ax.transAxes, fontsize=14, fontweight='bold', va='bottom')
    if subtitle:
        ax.text(0, 1.015, subtitle, transform=ax.transAxes, fontsize=10, color=MUTED, va='bottom')

def kmfmt(x, _pos=None):
    """大数字转 K / M（保留必要的一位小数，避免 1.5K 被取整成 2K 与 2K 重复）。"""
    if x >= 1e6:
        return f'{x/1e6:.1f}M'
    if x >= 1e3:
        return f'{x/1e3:.1f}'.rstrip('0').rstrip('.') + 'K'
    return f'{x:.0f}'

print('环境就绪  pandas', pd.__version__, '| numpy', np.__version__)

In [ ]:
# 数据源开关：USE_DB=True 直连 MySQL；=False 则读 reports/sql_outputs 下导出的 CSV（无需数据库密码）
USE_DB = True

PROJECT_DIR = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
CSV_DIR = PROJECT_DIR / 'reports' / 'sql_outputs'
FIG_DIR = PROJECT_DIR / 'reports' / 'figures'
FIG_DIR.mkdir(parents=True, exist_ok=True)

DB = dict(host='127.0.0.1', port=3306, user='root', database='ecommerce_analysis')
_engine = None

def _get_engine():
    global _engine
    if _engine is None:
        from sqlalchemy import create_engine
        pwd = os.environ.get('MYSQL_PWD') or getpass.getpass('MySQL 密码: ')
        url = f"mysql+pymysql://{DB['user']}:{pwd}@{DB['host']}:{DB['port']}/{DB['database']}?charset=utf8mb4"
        _engine = create_engine(url)
    return _engine

def load(table):
    """读一张分析结果表为 DataFrame。"""
    if USE_DB:
        return pd.read_sql(f'SELECT * FROM {table}', _get_engine())
    return pd.read_csv(CSV_DIR / f'{table}.csv')

def savefig(name):
    """把当前图保存到 reports/figures，供报告 / README 用。"""
    plt.savefig(FIG_DIR / f'{name}.png', bbox_inches='tight')

print('数据源:', 'MySQL' if USE_DB else f'CSV @ {CSV_DIR}')

## 1. 行为分布
各行为类型的事件量与占比，先看清数据的整体构成。

In [ ]:
ev = load('analysis_event_distribution').sort_values('events')
ev['event_share'] = ev['events'] / ev['events'].sum()
display(ev.sort_values('events', ascending=False).reset_index(drop=True))

fig, ax = plt.subplots(figsize=(8.5, 3.4))
colors = sns.color_palette('crest', len(ev))
ax.barh(ev['event_type'], ev['events'], color=colors, height=0.62, zorder=3)
for y, (v, s) in enumerate(zip(ev['events'], ev['event_share'])):
    ax.text(v, y, f'  {v:,.0f}   ({s:.1%})', va='center', ha='left', fontsize=10, color=INK)
ax.xaxis.set_major_formatter(mticker.FuncFormatter(kmfmt))
ax.set_xlim(0, ev['events'].max() * 1.2)
ax.grid(axis='y', visible=False)
sns.despine(ax=ax, left=True)
titled(ax, '各行为类型事件量', 'view / cart / purchase 三类行为的规模与占比')
plt.tight_layout(); savefig('01_event_distribution'); plt.show()

## 2. 转化漏斗
- **无序总览漏斗**（`funnel_overview`）：曾浏览 / 曾加购 / 曾购买，各独立人数。
- **用户级严格漏斗**（`funnel_user`）：按 view → cart → purchase 有序，更贴近真实路径转化。

> 注意：本数据集大量购买没有前置加购事件，所以严格漏斗的购买人数会明显低于实际买家数，解读时以\"加购路径转化\"理解。

In [ ]:
fo = load('analysis_funnel_overview').sort_values('stage_order')
fu = load('analysis_funnel_user').sort_values('stage_order')
print('—— 无序总览漏斗 ——'); display(fo)
print('—— 用户级严格漏斗 ——'); display(fu)

labels_cn = ['浏览 view', '加购 cart', '购买 purchase']
vals = fu['users'].tolist()

fig, ax = plt.subplots(figsize=(8.5, 4))
y = list(range(len(vals)))
ax.barh(y, vals, color=FUNNEL_COLORS, height=0.6, zorder=3)
ax.set_yticks(y); ax.set_yticklabels(labels_cn[:len(vals)]); ax.invert_yaxis()
for i, v in enumerate(vals):
    conv = '' if i == 0 else f'      ↓ 上一环转化 {vals[i] / vals[i-1]:.1%}'
    ax.text(v, i, f'  {v:,.0f}{conv}', va='center', ha='left', fontsize=10, color=INK)
ax.xaxis.set_major_formatter(mticker.FuncFormatter(kmfmt))
ax.set_xlim(0, vals[0] * 1.3)
ax.grid(axis='y', visible=False)
sns.despine(ax=ax, left=True)
overall = vals[-1] / vals[0] if vals[0] else 0
titled(ax, '用户转化漏斗  view → cart → purchase', f'整体转化率（浏览 → 购买）{overall:.2%}')
plt.tight_layout(); savefig('02_funnel_user'); plt.show()

## 3. 加购流失（辅助观察）
加购用户里有多少又把商品移出了购物车，以及移出后是否仍然购买。

> 本数据集（多品类商店）没有 `remove_from_cart` 行为，此段大概率为 0 / 空，属正常，可在结论中说明后跳过。

In [ ]:
cr = load('analysis_cart_remove_summary'); display(cr)
r = cr.iloc[0]
print(f"加购用户            : {r['cart_users']:,.0f}")
print(f"其中移出过购物车     : {r['cart_users_with_remove']:,.0f}  （占加购用户 {r['remove_among_cart_users_rate']:.1%}）")
print(f"移出后仍然购买       : {r['removed_but_purchased_users']:,.0f}")

## 4. 用户留存
按用户首次活跃日分组（cohort），看次日 / 7 日 / 30 日留存，以及 1-14 天留存热力图。

In [ ]:
rc = load('analysis_retention_cohort_summary').sort_values('cohort_date')
rc['cohort_date'] = pd.to_datetime(rc['cohort_date'])
display(rc.head())

fig, ax = plt.subplots(figsize=(11, 4.2))
series = [('d1_retention', '次日留存', PRIMARY), ('d7_retention', '7 日留存', ACCENT), ('d30_retention', '30 日留存', MUTED)]
for col, lab, c in series:
    ax.plot(rc['cohort_date'], rc[col], color=c, lw=2, marker='o', ms=3.5, label=lab)
    m = rc[col].mean()
    ax.axhline(m, color=c, lw=1, ls='--', alpha=0.35)
ax.yaxis.set_major_formatter(mticker.PercentFormatter(1.0))
ax.set_ylim(0, None)
ax.legend(loc='upper right', ncol=3)
sns.despine(ax=ax)
titled(ax, '用户留存随 cohort 的变化', '按首次活跃日分组的 D1 / D7 / D30 留存率（虚线为各自均值）')
plt.tight_layout(); savefig('03_retention_curve'); plt.show()

In [ ]:
rm = load('analysis_retention_matrix_30d')
rm['cohort_date'] = pd.to_datetime(rm['cohort_date']).dt.strftime('%m-%d')
pivot = rm.pivot(index='cohort_date', columns='retention_day', values='retention_rate')
cols = [c for c in pivot.columns if 1 <= c <= 14]  # 去掉第 0 天（恒为 100%），聚焦留存衰减
# 用数据 96 分位定色阶上限，避免个别离群 cohort 把整体压暗
vmax = max(5.0, float(np.nanpercentile(pivot[cols].values * 100, 96)))

fig, ax = plt.subplots(figsize=(11, 8.5))
sns.heatmap(pivot[cols] * 100, cmap='mako_r', vmin=0, vmax=vmax, ax=ax,
            linewidths=0.4, linecolor='white',
            cbar_kws={'label': '留存率 (%)', 'shrink': 0.5, 'pad': 0.02})
ax.set_xlabel('活跃后第 N 天'); ax.set_ylabel('首次活跃日')
ax.tick_params(axis='y', labelsize=7, rotation=0)
ax.tick_params(axis='x', labelsize=9)
ax.text(0, 1.02, '留存热力图（cohort × 活跃后第 N 天，1-14 天）',
        transform=ax.transAxes, fontsize=14, fontweight='bold', va='bottom')
plt.tight_layout(); savefig('04_retention_heatmap'); plt.show()

## 5. 复购留存
首次购买后 D1 / D7 / D30 是否再次购买。

In [ ]:
rp = load('analysis_repurchase_retention').sort_values('purchase_cohort_date')
rp['purchase_cohort_date'] = pd.to_datetime(rp['purchase_cohort_date'])
display(rp.head(10))

fig, ax = plt.subplots(figsize=(11, 4.2))
for col, lab, c in [('d1_repurchase_rate', 'D1 复购', PRIMARY), ('d7_repurchase_rate', 'D7 复购', ACCENT), ('d30_repurchase_rate', 'D30 复购', MUTED)]:
    ax.plot(rp['purchase_cohort_date'], rp[col], color=c, lw=2, marker='o', ms=3.5, label=lab)
ax.yaxis.set_major_formatter(mticker.PercentFormatter(1.0))
ax.set_ylim(0, None); ax.legend(loc='upper right', ncol=3)
sns.despine(ax=ax)
titled(ax, '复购留存随购买 cohort 的变化', '首次购买后 D1 / D7 / D30 再次购买的比例')
plt.tight_layout(); savefig('05_repurchase_curve'); plt.show()

print('整体平均：',
      'D1', f"{rp['d1_repurchase_rate'].mean():.2%}",
      '| D7', f"{rp['d7_repurchase_rate'].mean():.2%}",
      '| D30', f"{rp['d30_repurchase_rate'].mean():.2%}")

## 6. RFM 用户分层
各分层的用户规模与消费额贡献——找出\"人少但贡献大\"的高价值群体。

In [ ]:
seg = load('analysis_rfm_segment_summary').sort_values('segment_monetary')
display(seg.sort_values('segment_monetary', ascending=False).reset_index(drop=True))

fig, axes = plt.subplots(1, 2, figsize=(13, 4.8))
axes[0].barh(seg['user_segment'], seg['users'], color=sns.color_palette('crest', len(seg)), zorder=3)
for i, v in enumerate(seg['users']):
    axes[0].text(v, i, f'  {v:,.0f}', va='center', fontsize=9, color=INK)
axes[0].xaxis.set_major_formatter(mticker.FuncFormatter(kmfmt))
axes[0].set_xlim(0, seg['users'].max() * 1.18)
axes[0].grid(axis='y', visible=False); sns.despine(ax=axes[0], left=True)
titled(axes[0], '各 RFM 分层 · 用户数')

axes[1].barh(seg['user_segment'], seg['monetary_ratio'], color=sns.color_palette('flare', len(seg)), zorder=3)
for i, v in enumerate(seg['monetary_ratio']):
    axes[1].text(v, i, f'  {v:.1%}', va='center', fontsize=9, color=INK)
axes[1].xaxis.set_major_formatter(mticker.PercentFormatter(1.0))
axes[1].set_xlim(0, seg['monetary_ratio'].max() * 1.18)
axes[1].grid(axis='y', visible=False); sns.despine(ax=axes[1], left=True)
titled(axes[1], '各 RFM 分层 · 消费额占比')

plt.tight_layout(); savefig('06_rfm_segments'); plt.show()

print('—— RFM 原始指标概览 ——'); display(load('analysis_rfm_metric_summary'))

## 7. 结论与发现（边做边记，最终报告主线）

> 按 PROJECT.md 的约定，每条结论都要能指向后续模块（指标监控 / A·B 实验 / 流失预测）。

- **行为构成**：_（待填，如 view 占比、purchase 占比）_
- **转化瓶颈**：_（待填，哪一环掉得最狠 → 模块 3 A/B 实验的假设来源）_
- **留存表现**：_（待填，次日/7日留存水平，是否存在快速衰减）_
- **复购情况**：_（待填）_
- **高价值人群**：_（待填，哪个分层人少但贡献大 → 模块 4 流失预测重点保护对象）_